# S2.14 — Spark UI First Look
**Date completed:** September 2026  
**Status:** In Progress  
**Note:** Full Spark UI mastery in S13 — Performance Engineering

# Create the dataset

In [0]:
# ============================================================
# Cell 3 — Dataset Preparation
# Goal: Create a common dataset for all three experiments
# ============================================================

import pyspark.sql.functions as F
import time

df = spark.range(0, 10000000)

df = (
    df.withColumn(
        "city",
        F.when(F.col("id") % 4 == 0, "Delhi")
         .when(F.col("id") % 4 == 1, "Mumbai")
         .when(F.col("id") % 4 == 2, "Pune")
         .otherwise("Chennai")
    )
    .withColumn(
        "order_value",
        (F.col("id") % 1000).cast("double")
    )
)

print("Dataset prepared: 10 million rows")

# Narrow transformation

In [0]:
# ============================================================
# Cell 5 — Narrow Transformation
# Goal: Observe Filter + Project without an explicit shuffle
# ============================================================

print("=== NARROW TRANSFORMATION ===")

df_narrow = (
    df.filter(
        (F.col("id") < 4000) &
        (F.col("city") == "Delhi")
    )
    .select("id", "city", "order_value")
)

# Inspect physical execution plan
df_narrow.explain("formatted")

start = time.perf_counter()

# Small output — safe to collect for this experiment
rows = df_narrow.collect()

end = time.perf_counter()

print(f"Records returned: {len(rows)}")
print(f"Execution time: {end-start:.4f} seconds")

# Wide transformation

In [0]:
# ============================================================
# Cell 7 — Wide Transformation
# Goal: Inspect the shuffle caused by groupBy + aggregation
# ============================================================

print("=== WIDE TRANSFORMATION ===")

df_wide = df.groupBy("city").agg(
    F.count("*").alias("total_orders"),
    F.sum("order_value").alias("total_revenue"),
    F.avg("order_value").alias("avg_order")
)

df_wide.explain("formatted")

start = time.perf_counter()

df_wide.show()

end = time.perf_counter()

print(f"Execution time: {end-start:.4f} seconds")

# Explicit repartition

In [0]:
# ============================================================
# Cell 9 — Explicit Repartition
# Goal: Inspect the shuffle introduced by repartition(16)
# ============================================================

print("=== REPARTITION EXPERIMENT ===")

df_repartitioned = df.repartition(16)

df_result = df_repartitioned.agg(
    F.sum("order_value").alias("total_revenue")
)

df_result.explain("formatted")

start = time.perf_counter()

df_result.show()

end = time.perf_counter()

print(f"Execution time: {end-start:.4f} seconds")

In [0]:
# ============================================================
# Cell 10 — Avoid Unnecessary Sort and Shuffle
#
# Goal:
# 1. Remove explicit repartition(16).
# 2. Let Spark perform partial aggregation.
# 3. Compare Query Profile with the previous execution.
# ============================================================

import time
import pyspark.sql.functions as F

print("=== OPTIMISATION: WITHOUT REPARTITION ===")

# Step 1 — Define aggregation directly
df_optimized = df.agg(
    F.sum("order_value").alias("total_revenue")
)

# Step 2 — Inspect physical execution plan
df_optimized.explain("formatted")

# Step 3 — Execute
start = time.perf_counter()

df_optimized.show()

end = time.perf_counter()

print(f"Execution time: {end-start:.4f} seconds")

In [0]:
# ------------------------------------------------------------
# TEST 4 — Optimised SUM (No Explicit Repartition)
# ------------------------------------------------------------

start = time.perf_counter()

optimized_rows = df_optimized.collect()

optimized_time = time.perf_counter() - start

summary.append((
    "Optimised SUM",
    "Aggregation",
    "No explicit repartition",
    len(optimized_rows),
    optimized_time
))

In [0]:
# ============================================================
# Cell 11 — FINAL SPARK EXECUTION SUMMARY
#
# Goal:
# 1. Compare Narrow and Wide transformations.
# 2. Compare execution time.
# 3. Display output records.
# 4. Identify which operations introduce a shuffle.
#
# Prerequisite:
# Execute Cells 2A, 2B, 2C and 2D first.
# ============================================================

import time

print("=" * 75)
print("       SPARK EXECUTION — FINAL COMPARISON")
print("=" * 75)

summary = []

# ------------------------------------------------------------
# TEST 1 — Narrow Transformation
# ------------------------------------------------------------

start = time.perf_counter()

narrow_rows = df_narrow.collect()

narrow_time = time.perf_counter() - start

summary.append((
    "Filter + Select",
    "Narrow",
    "No explicit shuffle",
    len(narrow_rows),
    narrow_time
))

# ------------------------------------------------------------
# TEST 2 — Wide Transformation
# ------------------------------------------------------------

start = time.perf_counter()

wide_rows = df_wide.collect()

wide_time = time.perf_counter() - start

summary.append((
    "GroupBy + Aggregate",
    "Wide",
    "Yes",
    len(wide_rows),
    wide_time
))

# ------------------------------------------------------------
# TEST 3 — Explicit Repartition
# ------------------------------------------------------------

start = time.perf_counter()

repartition_rows = df_result.collect()

repartition_time = time.perf_counter() - start

summary.append((
    "Repartition + SUM",
    "Wide",
    "Yes",
    len(repartition_rows),
    repartition_time
))

# ------------------------------------------------------------
# FINAL COMPARISON TABLE
# ------------------------------------------------------------

print()
print(
    f"{'Operation':<23}"
    f"{'Type':<12}"
    f"{'Shuffle':<22}"
    f"{'Rows':<10}"
    f"{'Time (sec)':<12}"
)

print("-" * 79)

for operation, transformation, shuffle, rows, elapsed in summary:

    print(
        f"{operation:<23}"
        f"{transformation:<12}"
        f"{shuffle:<22}"
        f"{rows:<10}"
        f"{elapsed:<12.4f}"
    )

print("-" * 79)

# ------------------------------------------------------------
# LEARNING CONCLUSION
# ------------------------------------------------------------

print()
print("=== KEY OBSERVATIONS ===")

print("1. Filter + Select are narrow transformations.")
print("2. GroupBy + Aggregate generally requires shuffle.")
print("3. repartition(16) explicitly introduces shuffle.")
print("4. Shuffle redistributes data across partitions.")
print("5. Execution time alone does not prove shuffle cost.")
print("6. Verify actual operators using Query Profile.")

print()
print("=== EXPERIMENT COMPLETED ===")

%md
## Key Takeaways — S2.14 Spark UI First Look

## What We Saw in the Performance Panel
| Job | Tasks | Time | Type |
|-----|-------|------|------|
| narrow_rows (filter) | 8/8 | 151ms | Narrow — no shuffle |
| wide_rows (groupBy) | 9/9 | 309ms | Wide — shuffle |
| repartition_rows | 25/25 | 429ms | Wide — full data shuffle |

## Repartition Query Profile — Reading Bottom to Top
| Step | Operation | Rows | Time | Note |
|------|-----------|------|------|------|
| #8 | Range | 10M | 5ms | Generate data |
| #7 | Sort | 10M | 295ms | Most expensive — sorts before shuffle |
| #6 | Shuffle | 10M | 223ms | Full data redistribution — ALL rows move |
| #5 | Aggregate | 10M | 10ms | Partial sums per partition |
| #4 | Shuffle | 16 | 3ms | Tiny — partial sums only |
| #3 | Aggregate | 16 | 0ms | Final total |
| #1 | Result | 1 | 0ms | Answer returned |

## Key Findings
1. repartition shuffles ALL rows (10M) — very expensive
2. groupBy shuffles only partial counts (32 rows) — cheap
3. Sort happens BEFORE repartition shuffle — double cost
4. Two shuffle stages in repartition job — Sort + Shuffle
5. Narrow transformation (filter) is always fastest

## Spark UI Tabs — Full coverage in S13
- Jobs tab — all jobs and their status
- Stages tab — stages inside each job
- Tasks tab — individual task timelines
- SQL tab — query plan visualisation
- Storage tab — cached DataFrames
- Environment tab — all Spark configs